# Milestone 2: Clean and prepare the sales data

This notebook prepares a mini-market transactions for analysis. 

We will tidy the data, make repeated items count correctly, connect products, then calculate monthly and net sales.

## 1. Set up the notebook

We will use pandas to read and prepare the CSV files. The folder search lets this work when the notebook is opened from the project folder or its notebooks folder. This notebook is designed to be run from the project root folder, but it can also be run from the notebooks folder.

In [1]:
from pathlib import Path
import pandas as pd


ROOT = Path.cwd()
while not (ROOT / "data" / "raw" / "transactions_2025.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if not (ROOT / "data" / "raw" / "transactions_2025.csv").exists():
    raise FileNotFoundError("Open this notebook from the project folder so it can find data/raw/.")

RAW_DIR = ROOT / "data" / "raw"
REFERENCE_DIR = ROOT / "data" / "reference"
PROCESSED_DIR = ROOT / "data" / "processed"
REPORTS_DIR = ROOT / "reports"
quality = []

def record(step, before, after, changed, explanation, rows_removed=None, rows_combined=0):
    quality.append({
        "step": step,
        "rows_before": int(before),
        "rows_after": int(after),
        "rows_removed": int(before - after if rows_removed is None else rows_removed),
        "rows_combined": int(rows_combined),
        "values_fixed_or_flagged": int(changed),
        "explanation": explanation,
    })

def missing_text(values):
    return values.isna() | values.astype("string").str.strip().eq("").fillna(True)

print(f"Project folder:{ROOT}")

Project folder:c:\Users\Administrator\Documents\LUX CAPSTONE PROJECT


## 2. Load and preview the raw transactions

We first read every transaction field as text. That lets us convert dates and numbers deliberately in the next step. This won`t change the raw files, and we can check that the counts match the original CSVs.

In [2]:
transactions = pd.read_csv(RAW_DIR / "transactions_2025.csv", dtype="string")
required_columns = [
    "sale_date", "transaction_id", "store_id", "product_id",
    "quantity", "unit_price_kes", "promotion_pct", "payment_method",
]
missing_columns = [column for column in required_columns if column not in transactions.columns]
if missing_columns:
    raise ValueError(f"Transactions file is missing columns: {missing_columns}")

raw_row_count = len(transactions)
record("Raw input", raw_row_count, raw_row_count, 0, "Loaded the transaction CSV; raw files stay unchanged.")
transactions.head()

,sale_date,transaction_id,store_id,customer_id,product_id,quantity,unit_price_kes,promotion_pct,payment_method
0,2025-01-01,T00001,S001,<NA>,P002,2,75,0,Cash
1,2025-01-01,T00001,S001,<NA>,P009,2,60,5,Cash
2,2025-01-01,T00002,S001,C094,P008,1,150,0,M-Pesa
3,2025-01-01,T00002,S001,C094,P002,2,75,0,M-Pesa
4,2025-01-01,T00003,S002,C018,P001,3,180,0,M-Pesa


## 3. Convert types and handle missing values

Dates and amounts stored as text are hard to analyse, so we parse them. A blank customer ID is expected for a walk-in purchase, so we label it WALK_IN. We also check that the counts match the original CSVs, and that there are no missing required fields or impossible values. We will drop any rows that fail these checks, but there are none in this dataset.

In [3]:
for column in transactions.select_dtypes(include=["string"]).columns:
    transactions[column] = transactions[column].str.strip()

converted_values = 0
transactions["sale_date"] = pd.to_datetime(transactions["sale_date"], errors="coerce")
converted_values += int(transactions["sale_date"].notna().sum())
for column in ["quantity", "unit_price_kes", "promotion_pct"]:
    was_present = transactions[column].notna()
    parsed = pd.to_numeric(transactions[column], errors="coerce")
    converted_values += int((was_present & parsed.notna()).sum())
    transactions[column] = parsed

record(
    "Type conversion", len(transactions), len(transactions), converted_values,
    "Parsed sale_date as a date and quantity, price, and promotion as numbers.",
)

walk_in = missing_text(transactions["customer_id"])
missing_required = pd.Series(False, index=transactions.index)
for column in required_columns:
    if column in ["transaction_id", "store_id", "product_id", "payment_method"]:
        missing_required |= missing_text(transactions[column])
    else:
        missing_required |= transactions[column].isna()

before = len(transactions)
missing_required_count = int(missing_required.sum())
transactions = transactions.loc[~missing_required].copy()
transactions.loc[walk_in.loc[transactions.index], "customer_id"] = "WALK_IN"
walk_in_count = int(walk_in.loc[~missing_required].sum())
record(
    "Missing values", before, len(transactions), walk_in_count + missing_required_count,
    f"Filled {walk_in_count} blank customer IDs as WALK_IN; removed {missing_required_count} rows missing required details.",
    rows_removed=missing_required_count,
)
print(f"Converted {converted_values:,} values; labeled {walk_in_count:,} walk-in customer IDs.")

Converted 6,192 values; labeled 498 walk-in customer IDs.


## 4. Combine repeated items and check unusual values

The same product can appear twice on one receipt because a customer may buy more than one of it. We combine matching receipt lines by adding their quantities, rather than deleting them and losing units. Then we remove any rows with impossible values, such as negative quantities or prices. There are none in this dataset.

In [4]:
before = len(transactions)
units_before = int(transactions["quantity"].sum())
grouping_columns = [column for column in transactions.columns if column != "quantity"]
transactions = transactions.groupby(
    grouping_columns, as_index=False, dropna=False, sort=False
)["quantity"].sum()
combined_rows = before - len(transactions)
if int(transactions["quantity"].sum()) != units_before:
    raise ValueError("Combining receipt items changed the total units sold.")
record(
    "Repeated receipt items", before, len(transactions), combined_rows,
    "Combined matching items by summing quantity; all units were preserved.",
    rows_removed=0, rows_combined=combined_rows,
)

before = len(transactions)
outlier = (
    (transactions["quantity"] < 1) | (transactions["quantity"] > 100)
    | (transactions["unit_price_kes"] <= 0) | (transactions["unit_price_kes"] > 100_000)
    | (transactions["promotion_pct"] < 0) | (transactions["promotion_pct"] > 100)
)
outlier_count = int(outlier.sum())
transactions = transactions.loc[~outlier].copy()
record(
    "Outlier and range checks", before, len(transactions), outlier_count,
    "Removed rows outside quantity 1-100, price greater than KES 0 up to KES 100,000, or promotion 0-100%.",
    rows_removed=outlier_count,
)
print(f"Repeated item lines combined: {combined_rows:,}; units before and after combining: {units_before:,}.")

Repeated item lines combined: 37; units before and after combining: 2,242.


## 5. Add useful names and sales features

We connect the IDs to store, product and customers. Then we create a month field and calculate gross sales, discount, and net sales for each product line. Net sales are the amount after the promotion discount. We check that the total net sales in the cleaned CSV matches the raw CSV, and that the cleaned CSV has no missing required fields or impossible values.

In [5]:
stores = pd.read_csv(RAW_DIR / "stores.csv", dtype="string")
products = pd.read_csv(RAW_DIR / "products.csv", dtype="string")
customers = pd.read_csv(RAW_DIR / "customers.csv", dtype="string")
population = pd.read_csv(REFERENCE_DIR / "knbs_county_population_2019.csv", dtype="string")

transactions = transactions.merge(
    stores[["store_id", "store_name", "county", "location"]],
    on="store_id", how="left", validate="many_to_one",
)
transactions = transactions.merge(
    products[["product_id", "product_name", "category"]],
    on="product_id", how="left", validate="many_to_one",
)
transactions = transactions.merge(
    customers[["customer_id", "customer_type"]],
    on="customer_id", how="left", validate="many_to_one",
)
transactions = transactions.merge(
    population[["county", "population_2019"]],
    on="county", how="left", validate="many_to_one",
)
transactions["customer_type"] = transactions["customer_type"].fillna("Walk-in / not identified")
transactions["population_2019"] = pd.to_numeric(transactions["population_2019"], errors="coerce")

joined_columns = ["store_name", "county", "population_2019", "product_name"]
if transactions[joined_columns].isna().any().any():
    raise ValueError("A store, product, or county population did not match its reference file.")

transactions["sale_month"] = transactions["sale_date"].dt.strftime("%Y-%m")
transactions["gross_sales_kes"] = transactions["quantity"] * transactions["unit_price_kes"]
transactions["discount_kes"] = transactions["gross_sales_kes"] * transactions["promotion_pct"] / 100
transactions["net_sales_kes"] = transactions["gross_sales_kes"] - transactions["discount_kes"]
for column in ["gross_sales_kes", "discount_kes", "net_sales_kes"]:
    transactions[column] = transactions[column].round(2)

if (transactions["net_sales_kes"] < 0).any():
    raise ValueError("A net sales value is negative after cleaning.")

output_columns = [
    "sale_date", "sale_month", "transaction_id", "store_id", "store_name",
    "county", "population_2019", "location", "customer_id", "customer_type",
    "product_id", "product_name", "category", "quantity", "unit_price_kes",
    "promotion_pct", "gross_sales_kes", "discount_kes", "net_sales_kes", "payment_method",
]
transactions = transactions[output_columns].sort_values(["sale_date", "transaction_id", "product_id"])
transactions.head()

,sale_date,sale_month,transaction_id,store_id,store_name,county,population_2019,location,customer_id,customer_type,product_id,product_name,category,quantity,unit_price_kes,promotion_pct,gross_sales_kes,discount_kes,net_sales_kes,payment_method
0,2025-01-01,2025-01,T00001,S001,Sample CBD Mini-Mart,Nairobi City,4397073,Nairobi CBD,WALK_IN,Walk-in / not identified,P002,Milk 1L,Dairy,2,75,0,150,0.0,150.0,Cash
1,2025-01-01,2025-01,T00001,S001,Sample CBD Mini-Mart,Nairobi City,4397073,Nairobi CBD,WALK_IN,Walk-in / not identified,P009,Water 1.5L,Beverages,2,60,5,120,6.0,114.0,Cash
3,2025-01-01,2025-01,T00002,S001,Sample CBD Mini-Mart,Nairobi City,4397073,Nairobi CBD,C094,loyalty_member,P002,Milk 1L,Dairy,2,75,0,150,0.0,150.0,M-Pesa
2,2025-01-01,2025-01,T00002,S001,Sample CBD Mini-Mart,Nairobi City,4397073,Nairobi CBD,C094,loyalty_member,P008,Tea leaves 250g,Beverages,1,150,0,150,0.0,150.0,M-Pesa
4,2025-01-01,2025-01,T00003,S002,Sample Ruiru Mini-Mart,Kiambu,2417735,Ruiru,C018,regular,P001,Maize flour 2kg,Staples,3,180,0,540,0.0,540.0,M-Pesa


## 6. Save the prepared data and review the evidence

The notebook saves a clean CSV and a small quality report. Re-running these cells overwrites those generated outputs, while leaving the original data untouched.

In [7]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
clean_file = PROCESSED_DIR / "clean_sales_2025.csv"
quality_file = REPORTS_DIR / "quality_report.csv"
transactions.to_csv(clean_file, index=False, date_format="%Y-%m-%d")
quality_report = pd.DataFrame(quality)
quality_report.to_csv(quality_file, index=False)

assert transactions["sale_date"].notna().all()
assert transactions["customer_id"].notna().all()
assert transactions["product_name"].notna().all()
assert transactions["store_name"].notna().all()
assert (transactions["quantity"] > 0).all()
assert (transactions["net_sales_kes"] >= 0).all()

print(f"Prepared {len(transactions):,} rows from {raw_row_count:,} raw rows.")
print(f"Clean data saved to: {clean_file.relative_to(ROOT)}")
print(f"Quality report saved to: {quality_file.relative_to(ROOT)}")
quality_report

Prepared 1,511 rows from 1,548 raw rows.
Clean data saved to: data\processed\clean_sales_2025.csv
Quality report saved to: reports\quality_report.csv


,step,rows_before,rows_after,rows_removed,rows_combined,values_fixed_or_flagged,explanation
0,Raw input,1548,1548,0,0,0,Loaded the transaction CSV; raw files stay unc...
1,Type conversion,1548,1548,0,0,6192,"Parsed sale_date as a date and quantity, price..."
2,Missing values,1548,1548,0,0,498,Filled 498 blank customer IDs as WALK_IN; remo...
3,Repeated receipt items,1548,1511,0,37,37,Combined matching items by summing quantity; a...
4,Outlier and range checks,1511,1511,0,0,0,"Removed rows outside quantity 1-100, price gre..."
